## Preprocess BactHeCom DB

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3

import warnings
warnings.filterwarnings('ignore')

In [2]:
# paths
data_dir = os.path.join("..", "data")
results_dir = os.path.join("..", "results")
os.makedirs(results_dir, exist_ok=True)

bacthecom_db=f"{data_dir}/db_bacthecom.db"

In [3]:
## Load database 
conn = sqlite3.connect(bacthecom_db)

tbls = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table';", con=conn)
tbls = tbls[tbls['name'] != 'sqlite_sequence']
print(f"Tables in the database: {tbls['name'].values}")

Tables in the database: ['paciente' 'factores_riesgo_infeccion_bmr' 'antibiograma'
 'semantic_mapping' 'episodio_infeccion' 'comorbilidad' 'signos_sintomas'
 'episodio_ingreso']


In [4]:
for tbl in tbls['name']:
    if tbl == "sqlite_sequence":
        continue    
    print(f"{tbl}")
    df = pd.read_sql_query(f"SELECT * FROM {tbl} LIMIT 5;", con=conn)
    print(f"Variables: {df.columns.values}")
    print("\n") 

paciente
Variables: ['record_id' 'sexo' 'fecha_nacimiento']


factores_riesgo_infeccion_bmr
Variables: ['record_id' 'fecha_ingreso' 'hospit_ano_previo' 'hospit_mes_previo'
 'hospit_ano_previo_uci' 'hemodialisis_permanente' 'dialisis_peritoneal'
 'cateter_venoso' 'sonda_urinaria' 'sonda_nasogastrica'
 'derivacion_ventriculoper' 'valvula_prot_cardiaca'
 'portador_otros_disposit']


antibiograma
Variables: ['episode_id' 'antimicrobiano' 'cmi' 'interpretacion']


semantic_mapping
Variables: ['table_name' 'variable_name' 'description' 'uri']


episodio_infeccion
Variables: ['episode_id' 'record_id' 'fecha_ingreso' 'fecha_cultivo' 'area_hosp'
 'id_cultivo' 'especimen' 'microorganismo' 'fenotipo_resistencia']


comorbilidad
Variables: ['record_id' 'fecha_ingreso' 'infarto' 'insuficiencia_cardiaca' 'evp'
 'e_cerebrovascular' 'demencia' 'e_pulmonar_cronica' 'ulcera_peptica'
 'colagenopatia' 'hemiplejia' 'erc' 'neoplasia_tratamiento_activo'
 'neoplasia_solida_metastasica' 'neoplasia_solida_no_me

### tbl_pacientes

In [5]:
tbl_pacientes = pd.read_sql_query("SELECT * FROM paciente;", con=conn)
tbl_pacientes.head()

,record_id,sexo,fecha_nacimiento
0,1,Hombre,1943-03-28
1,2,Mujer,1959-12-16
2,3,Hombre,1976-11-14
3,4,Hombre,1992-09-27
4,5,Mujer,1994-07-13


In [6]:
# Recode 'Hombre'/'Mujer' to 0/1
tbl_pacientes['sexo'] = tbl_pacientes['sexo'].map({'Hombre': 0, 'Mujer': 1})

### tbl_factores_riesgo_infeccion_bmr

In [7]:
tbl_factores_bmr = pd.read_sql_query("SELECT * FROM factores_riesgo_infeccion_bmr;", con=conn)
tbl_factores_bmr.head() 

,record_id,fecha_ingreso,hospit_ano_previo,hospit_mes_previo,hospit_ano_previo_uci,hemodialisis_permanente,dialisis_peritoneal,cateter_venoso,sonda_urinaria,sonda_nasogastrica,derivacion_ventriculoper,valvula_prot_cardiaca,portador_otros_disposit
0,1,2021-08-24,0,0,0,0,0,0,0,1,0,0,0
1,2,2023-06-18,0,0,0,0,0,0,0,1,0,0,0
2,3,2022-02-11,1,0,1,0,0,0,0,1,0,0,0
3,4,2021-05-15,0,0,0,0,0,0,0,1,0,0,0
4,5,2021-07-04,0,0,0,0,0,0,0,0,0,0,0


### tbl_episodios

In [8]:
tbl_episodios = pd.read_sql_query("SELECT * FROM episodio_ingreso;", con=conn)
tbl_episodios.fillna(0, inplace=True)
tbl_episodios.head()

,record_id,fecha_ingreso,fecha_alta,organo_aparato,foco_controlable,IRAs_nosocomial,mortalidad,uci_por_el_episodio,duracion_UCI,mujer_gestante,codigo_postal,paciente_residencia
0,1,2021-08-24,2021-09-09,0,0.0,Si,1,1,14,0,4007,0.0
1,2,2023-06-18,2023-07-13,0,0.0,Si,0,1,3,0,4740,0.0
2,3,2022-02-11,2022-04-06,Infeccion de cateter vascular,1.0,Si,0,1,24,0,4700,0.0
3,4,2021-05-15,2021-07-28,0,0.0,Si,0,1,42,0,18800,0.0
4,5,2021-07-04,2022-01-04,Infeccion tracto respiratorio inferior,0.0,Si,0,1,78,0,23009,0.0


In [9]:
# Recode IRAs_nosocomial to '0/1'
tbl_episodios['IRAs_nosocomial'] = tbl_episodios['IRAs_nosocomial'].map({'No': 0, 'Si': 1})

In [10]:
# pd.get_dummies(tbl_episodios['organo_aparato'], dtype=int).head()

In [11]:
infecname_to_infeccode = {
    'infec_via_urinaria_superior': 'Infección de la vía urinaria superior',
    'fiebre_sin_foco': 'Fiebre sin foco',
    'infec_cateter_vascular': 'Infeccion de cateter vascular',
    'infec_vias_biliares': 'Infeccion vias biliares',
    'infec_intraabdominal': 'Infeccion intraabdominal',
    'infec_tracto_respiratorio_inferior': 'Infeccion tracto respiratorio inferior',
    'infec_piel_partes_blandas': 'Infeccion de piel y partes blandas',
    'infec_cardiovascular': 'Infeccion cardiovascular',
    'infec_osteoarticular': 'Infeccion osteoarticular',
    'etiologia_incierta': 'Otros/Infeccion de etiologia incierta',
    'infec_snc': 'Infeccion del SNC',
    'infec_genital': 'Infeccion genital'
}


In [12]:
# rename organo_aparato
tbl_episodios['organo_aparato'] = tbl_episodios['organo_aparato'].map({v: k for k, v in infecname_to_infeccode.items()})

# One-hot encode the 'organo_aparato' column
tbl_episodios_recoded = pd.get_dummies(tbl_episodios, columns=['organo_aparato'], dtype=int)

tbl_episodios_recoded.head()


,record_id,fecha_ingreso,fecha_alta,foco_controlable,IRAs_nosocomial,mortalidad,uci_por_el_episodio,duracion_UCI,mujer_gestante,codigo_postal,...,organo_aparato_infec_cardiovascular,organo_aparato_infec_cateter_vascular,organo_aparato_infec_genital,organo_aparato_infec_intraabdominal,organo_aparato_infec_osteoarticular,organo_aparato_infec_piel_partes_blandas,organo_aparato_infec_snc,organo_aparato_infec_tracto_respiratorio_inferior,organo_aparato_infec_via_urinaria_superior,organo_aparato_infec_vias_biliares
0,1,2021-08-24,2021-09-09,0.0,1,1,1,14,0,4007,...,0,0,0,0,0,0,0,0,0,0
1,2,2023-06-18,2023-07-13,0.0,1,0,1,3,0,4740,...,0,0,0,0,0,0,0,0,0,0
2,3,2022-02-11,2022-04-06,1.0,1,0,1,24,0,4700,...,0,1,0,0,0,0,0,0,0,0
3,4,2021-05-15,2021-07-28,0.0,1,0,1,42,0,18800,...,0,0,0,0,0,0,0,0,0,0
4,5,2021-07-04,2022-01-04,0.0,1,0,1,78,0,23009,...,0,0,0,0,0,0,0,1,0,0


### tbl_comorbilidad
- Pivotar *tipos de cancer* en columnas y codificar **0/1**
- Pivotar *tipos de hepatopatias* en columnas y codificar **0/1**

In [13]:
tbl_comorbilidades = pd.read_sql_query("SELECT * FROM comorbilidad;", con=conn)
tbl_comorbilidades.head()

,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,diabetes_sin_lesion_organo_diana,diabetes_con_lesion_organo_diana,inmunosupresion,causa_inmunosupresion,fecha_TOS,TOS,fecha_TPH,TPH,clasificacion_quemadura,gran_quemado
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,"T20.29XA, T21.22XA, T21.24XA, T24.291A, T22.29...",1
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,0,1,1,58606001,None,0,None,0,None,0


**TIPOS_CANCER**

In [14]:
# get unique cancer types and pivot to columns
types_cancer = (tbl_comorbilidades['tipo_cancer']
                       .dropna()
                       .apply(lambda x: ['tipo_cancer_' + x.split('.')[0].strip() for x in x.split(',')])
                       .explode()
                       .unique())

# create cancer type columns and initialize to 0
tbl_comorbilidades[types_cancer] = 0

# iterate over rows and set cancer type columns to 1 where applicable (comma-separated values)
for x,y in tbl_comorbilidades.iterrows():
    if pd.isna(y['tipo_cancer']):
        continue
    cancers = ['tipo_cancer_' + c.split('.')[0].strip() for c in y['tipo_cancer'].split(',')]
    for cancer in cancers:
        tbl_comorbilidades.at[x, cancer] = 1
if 'tipo_cancer' in tbl_comorbilidades.columns:
    del tbl_comorbilidades['tipo_cancer']
tbl_comorbilidades.head()

,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,tipo_cancer_D10,tipo_cancer_C51,tipo_cancer_D22,tipo_cancer_C47,tipo_cancer_D43,tipo_cancer_D15,tipo_cancer_C94,tipo_cancer_D39,tipo_cancer_C75,tipo_cancer_C54
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**TIPOS_HEPATOPATIAS**

In [15]:
types_hepatopatias = (tbl_comorbilidades['tipo_hepatopatia']
                      .dropna()
                      .apply(lambda x: ['tipo_hepatopatia_' + x.split('.')[0].strip() for x in x.split(',')])
                      .explode()
                    .unique())
tbl_comorbilidades[types_hepatopatias] = 0

# pivot hepatopatias to columns
for x,y in tbl_comorbilidades.iterrows():
    if pd.isna(y['tipo_hepatopatia']):
        continue
    hepatopatias=['tipo_hepatopatia_' + h.split('.')[0].strip() for h in y['tipo_hepatopatia'].split(',')]
    for hepato in hepatopatias:
        tbl_comorbilidades.at[x, hepato] = 1

if 'tipo_hepatopatia' in tbl_comorbilidades.columns:
    del tbl_comorbilidades['tipo_hepatopatia']
tbl_comorbilidades.head()

,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,tipo_hepatopatia_K74,tipo_hepatopatia_K70,tipo_hepatopatia_K71,tipo_hepatopatia_K73,tipo_hepatopatia_K75,tipo_hepatopatia_B18,tipo_hepatopatia_B19,tipo_hepatopatia_B17,tipo_hepatopatia_B16,tipo_hepatopatia_K77
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# # delete causa inmunosupresion (too much granularity)
# if 'causa_inmunosupresion' in tbl_comorbilidades.columns:
#     del tbl_comorbilidades['causa_inmunosupresion']

### tbl_signos

In [17]:
tbl_signos = pd.read_sql_query("SELECT * FROM signos_sintomas;", con=conn)
tbl_signos.head()

,record_id,fecha_ingreso,foco,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,situacion_funcional_basal,indice_de_charlson,escala_karnofsky,...,lesiones_piel,lesiones_mucosas,cefalea,dolores_articulares,temperatura,frec_cardiaca,frecuencia_respiratoria,tension_arterial_sist,tension_arterial_diast,saturacion_pO2
0,1,2021-08-24,None,1,1,0,None,None,11,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2023-06-18,None,0,0,0,None,None,4,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2022-02-11,cateter venoso,0,0,0,None,None,2,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2021-05-15,None,1,1,0,None,Necesita ayuda importante y asistencia médica ...,0,50.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2021-07-04,pulmonar,0,0,0,None,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


### tbl_infeccion

In [18]:
tbl_infeccion = pd.read_sql_query("SELECT * FROM episodio_infeccion;", con=conn)
tbl_infeccion.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,area_hosp,id_cultivo,especimen,microorganismo,fenotipo_resistencia
0,1,1,2021-08-24,2021-08-31,UCI,102007829,Sangre,Pseudomonas aeruginosa,None
1,2,2,2023-06-18,2023-06-30,General,103800271,Sangre,Escherichia coli,None
2,3,3,2022-02-11,2021-04-23,UCI,None,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE
3,4,3,2022-02-11,2021-04-23,UCI,None,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None
4,5,3,2022-02-11,2021-04-29,UCI,None,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None


In [19]:
pd.get_dummies(tbl_signos['foco'], prefix='foco', dtype=int).head()

,foco_ORL,foco_SNC,foco_biliar,foco_cardiovascular,foco_cateter venoso,foco_desconocido,foco_intraabdominal,foco_obstetricia/ginecológico,foco_odontogeno,foco_osteoarticular,foco_piel y partes blandas,foco_pulmonar,foco_urinario
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,1,0
